# Phase 4 — Model B training
Run on a Kaggle GPU. This notebook never loads ViLexNorm Test.

In [ ]:
!pip install -q -r requirements-kaggle.txt

In [ ]:
import platform, torch, transformers
print(platform.python_version(), torch.__version__, transformers.__version__)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'

## Paths
Update these three input paths to match the attached Kaggle datasets.

In [ ]:
from pathlib import Path
REPO = Path('/kaggle/working/VisolexNorm')
DATA = Path('/kaggle/input/visolexnorm-phase3')
MODEL_A = Path('/kaggle/input/visolexnorm-model-a/model_a')
WORK = Path('/kaggle/working')

In [ ]:
%cd {REPO}
!python -m pytest tests/unit/test_model_b_mixture.py tests/contract/test_model_b_contracts.py -q
!python scripts/build_model_b_mixture.py --repo-root . --phase3-manifest outputs/phase3_manifest.json --output /kaggle/working/training_mixture_manifest.json

## Smoke gate: exactly 200 gold + 200 pseudo

In [ ]:
!python scripts/train_model_b.py --model-a-checkpoint {MODEL_A} --data-dir {DATA} --mixture-manifest /kaggle/working/training_mixture_manifest.json --work-dir {WORK}/smoke --smoke-test
import json
smoke = json.load(open(WORK/'smoke/outputs/model_b/smoke_test.json'))
assert smoke['passed'] and smoke['composition'] == {'gold': 200, 'pseudo': 200}
smoke

## Full three-epoch run

In [ ]:
!python scripts/train_model_b.py --model-a-checkpoint {MODEL_A} --data-dir {DATA} --mixture-manifest /kaggle/working/training_mixture_manifest.json --work-dir {WORK}

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/model_b_artifacts', 'zip', '/kaggle/working', 'outputs/model_b')
shutil.make_archive('/kaggle/working/model_b_checkpoint', 'zip', '/kaggle/working', 'checkpoints/model_b')
print('Download both zip files from Kaggle Output.')